- "Improcedente - Dados incorretos/insuficientes"
- "Improcedente - Produção não contemplada para o especialista"
- "Improcedente - Seguimento do cliente não contemplado para o especialista"
- "Improcedente - Produção foi efetivada no mês anterior"
- "Improcedente - Produção será efetivada no proximo mês"
- "Improcedente - Outro angariador"
- "Procedente - Ajuste será realizado"


In [1]:
import random
import string
from datetime import datetime, timedelta
import pandas as pd
import re
import numpy as np

In [2]:
def capturarContrato(x):
    contrato = re.findall(r'66\d+', x)
    return contrato[0] if contrato else None

In [3]:
def verificarGradePesos(x):
    dfGrade = pd.read_excel("./database/gold/gradePesos.xlsx")
    
    # Adicione estas duas linhas para forçar a tipagem para string
    dfGrade["modelo"] = dfGrade["modelo"].astype(str)
    dfGrade["pin"] = dfGrade["pin"].astype(str)
    
    mask = (
        (dfGrade["modelo"] == str(x["CD_MODELO"])) &
        (dfGrade["pin"] == str(x["CD_PIN"]))
    )
    return not dfGrade.loc[mask].empty

In [4]:
def verificarModelo(x):
    dfModelo = pd.read_excel("./database/gold/modeloAngariacao.xlsx")
    
    # Forçando tipagem para string
    dfModelo["modelo"] = dfModelo["modelo"].astype(str)
    dfModelo["segmento"] = dfModelo["segmento"].astype(str)
    
    mask = (
        (dfModelo["modelo"] == str(x["CD_MODELO"])) &
        (dfModelo["segmento"] == str(x["CD_SEGMENTO"]))
    )
    return not dfModelo.loc[mask].empty

In [5]:
def forceType(df, t):
    for column in df.columns:
        df[column] = df[column].astype(t)

    return df


In [7]:
# Converte para DataFrame e sobrescreve o arquivo Bronze
# Obrigatório: o motorCorpao() realiza o read_excel na raiz física, logo precisamos salvar
df = pd.read_excel("./database/Bronze/chamadosTreinamento.xlsx")
forceType(df, str)
# Instancia o dataframe do motor com base nos dados atualizados
df_corpao = pd.read_excel("./database/Bronze/corpaoTreinamento.xlsx")
forceType(df_corpao, str)

# =============================================================================
# ETAPA 2: TRATAMENTO E CRUZAMENTO (LEFT JOIN)
# =============================================================================
# Extrai o contrato limpo no lado esquerdo para assegurar que a chave de cruzamento bata
df['regexContrato'] = df['NR_CONTRATO'].apply(lambda x: capturarContrato(str(x)))

# Left Join: Mantém todo o ruído à esquerda e traz as chaves de validação do corpão
df_chamados_catalogados = pd.merge(
    df,
    df_corpao[['regexContrato', 'CPF_CNPJ', 'CD_ANG', 'ANOMES']].rename(columns={'ANOMES': 'ANOMES_CORPAO'}),
    on=['regexContrato', 'CPF_CNPJ'],
    how='left'
)

# =============================================================================
# ETAPA 3: COLUNAS AUXILIARES BOOLEANAS (VALIDAÇÕES DE REGRA DE NEGÓCIO)
# =============================================================================
# 1. Valida estrutura do contrato (inicia e contém '66' + dígitos)
df_chamados_catalogados['valido_valor_contrato'] = df_chamados_catalogados['NR_CONTRATO'].apply(
    lambda x: bool(re.search(r'66\d+', str(x)))
)

# 2. Valida cliente (precisa ter exatamente 8 caracteres alfanuméricos)
df_chamados_catalogados['valido_numero_cliente'] = df_chamados_catalogados['CD_CLI'].apply(
    lambda x: bool(re.fullmatch(r'[A-Za-z0-9]{8}', str(x)))
)

# 3. Valida contemplação de modelo (aproveitando as funções base do seu pipeline)
df_chamados_catalogados['valido_modelo_contemplado'] = df_chamados_catalogados.apply(verificarGradePesos, axis=1)

# 4. Valida contemplação de segmento
df_chamados_catalogados['valido_seguimento_contemplado'] = df_chamados_catalogados.apply(verificarModelo, axis=1)

# 5. Valida unidade vs angariador (Se CD_ANG for nulo por conta de uma falha de join, resultará em False)
df_chamados_catalogados['valido_solicitante_angariador'] = (
    df_chamados_catalogados['CD_UNIDADE'] == df_chamados_catalogados['CD_ANG']
)

# =============================================================================
# ETAPA 4: MOTOR DE CLASSIFICAÇÃO (STATUS DO CHAMADO)
# =============================================================================
mes_atual = "202604"

# Condições funcionam em cascata. Os ruídos estruturais mais críticos barram 
# o chamado antes que a lógica caia em quebras temporais, garantindo robustez.
condicoes = [
    # Falha severa (dados sujos gerados na função de ruído)
    (~df_chamados_catalogados['valido_valor_contrato']) | (~df_chamados_catalogados['valido_numero_cliente']),
    
    # Validações de Produto/Negócio
    (~df_chamados_catalogados['valido_modelo_contemplado']),
    (~df_chamados_catalogados['valido_seguimento_contemplado']),
    
    # Falha limpa no Join (O dataframe da direita filtrou a informação)
    (df_chamados_catalogados['CD_ANG'].isna()),
    
    # Validações Temporais do Pipeline
    (df_chamados_catalogados['ANOMES_CORPAO'] < mes_atual),
    (df_chamados_catalogados['ANOMES_CORPAO'] > mes_atual),
    
    # Validação de Autoridade
    (~df_chamados_catalogados['valido_solicitante_angariador'])
]

escolhas = [
    "Improcedente - Dados incorretos/insuficientes",
    "Improcedente - Produção não contemplada para o especialista",
    "Improcedente - Seguimento do cliente não contemplado para o especialista",
    "Improcedente - Dados incorretos/insuficientes",
    "Improcedente - Produção foi efetivada no mês anterior",
    "Improcedente - Produção será efetivada no proximo mês",
    "improcedente - Outro angariador"
]

# Aplica a lógica com performance usando Numpy
df_chamados_catalogados['status_classificacao'] = np.select(
    condicoes,
    escolhas,
    default="Procedente - Ajuste será realizado"
)

# =============================================================================
# CONFERÊNCIA
# =============================================================================
display(df_chamados_catalogados.head())

print("\n--- Balanceamento das Classificações ---")
print(df_chamados_catalogados['status_classificacao'].value_counts())

df_chamados_catalogados.to_excel("./database/silver/chamados_catalogados.xlsx")

,Unnamed: 0,ID_TASK,CD_UNIDADE,CD_PIN,CD_MODELO,NR_CONTRATO,CD_CLI,CD_SEGMENTO,CPF_CNPJ,ANOMES,regexContrato,CD_ANG,ANOMES_CORPAO,valido_valor_contrato,valido_numero_cliente,valido_modelo_contemplado,valido_seguimento_contemplado,valido_solicitante_angariador,status_classificacao
0,0,9435,705384,3000,90,660001533924,18206649,256,28.316.518/8118-15,202604,660001533924,NaN,NaN,True,True,False,True,False,Improcedente - Produção não contemplada para o...
1,1,9436,922758,3000,251,660004857313,93598488,9,196.133.466-67,202604,660004857313,NaN,NaN,True,True,False,False,False,Improcedente - Produção não contemplada para o...
2,2,9437,635043,2051,946,660001661058,83552268,9,935.751.265-27,202604,660001661058,635043,202605,True,True,True,True,True,Improcedente - Produção será efetivada no prox...
3,3,9438,569500,3000,946,660007852826,79030938,11,251.318.290-05,202604,660007852826,569500,202604,True,True,True,True,True,Procedente - Ajuste será realizado
4,4,9439,389107,3054,201,0331250660008372459,32615647,9,604.434.057-71,202604,660008372459,NaN,NaN,True,True,False,False,False,Improcedente - Produção não contemplada para o...



--- Balanceamento das Classificações ---
status_classificacao
Improcedente - Produção não contemplada para o especialista                 2058
Improcedente - Dados incorretos/insuficientes                                833
Improcedente - Seguimento do cliente não contemplado para o especialista     704
Improcedente - Produção foi efetivada no mês anterior                        468
Improcedente - Produção será efetivada no proximo mês                        460
Procedente - Ajuste será realizado                                           382
improcedente - Outro angariador                                               95
Name: count, dtype: int64
